# 11 · Crop Monitoring and Agriculture

Monitor agricultural land, crop types, and vegetation health.

## Contents
1. Crop type mapping pipeline
2. NDVI time-series analysis
3. Crop health assessment
4. Phenology monitoring
5. Irrigation detection
6. Integration with field data

## 1 · Crop Type Mapping

In [ ]:
import pygeovision as pgv
import numpy as np

client = pgv.PyGeoVision()

BBOX = (-99.8, 41.2, -99.4, 41.5)     # Nebraska Corn Belt

# Use the crop type mapping end-to-end pipeline
result = client.pipeline(
    "crop_type_mapping",
    bbox=BBOX,
    output_dir="./results/crops/",
    date="2024-07",
    num_classes=13,
)

print(f"Crop Type Mapping: {'[OK]' if result.success else '[FAIL]'}")
if result.success:
    print(f"  Output: {result.output_path}")

# Individual crop types (Iowa/Nebraska example)
CROP_CLASSES = {
    0: "Background",
    1: "Corn",
    2: "Soybeans",
    3: "Cotton",
    4: "Rice",
    5: "Sorghum",
    6: "Winter Wheat",
    7: "Spring Wheat",
    8: "Oats",
    9: "Potatoes",
    10: "Vegetables",
    11: "Orchards",
    12: "Other Crops",
}
print("\nCrop type classes (CDL-compatible):")
for code, name in list(CROP_CLASSES.items())[:8]:
    print(f"  {code:2d}: {name}")

## 2 · NDVI Time-Series Analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulate an NDVI time series for a corn field (typical phenology)
# In practice: download multiple Sentinel-2 scenes and extract NDVI

months = np.arange(1, 13)
# Nebraska corn phenology
corn_ndvi = [0.15, 0.18, 0.25, 0.30, 0.55, 0.80, 0.87, 0.82, 0.65, 0.40, 0.20, 0.15]
soy_ndvi  = [0.15, 0.17, 0.22, 0.28, 0.45, 0.72, 0.80, 0.85, 0.70, 0.45, 0.22, 0.15]

month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(months, corn_ndvi, "o-", label="Corn",     color="#E8A838", linewidth=2.5, markersize=8)
ax.plot(months, soy_ndvi,  "s-", label="Soybeans", color="#5C9E31", linewidth=2.5, markersize=8)
ax.axhspan(0.8, 1.0, alpha=0.08, color="green", label="Peak growth")
ax.set_xlabel("Month", fontsize=13)
ax.set_ylabel("NDVI", fontsize=13)
ax.set_title("Crop Phenology: NDVI Time Series (Nebraska 2024)", fontsize=14, fontweight="bold")
ax.set_xticks(months); ax.set_xticklabels(month_labels)
ax.set_ylim(0, 1)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# Add growth stages for corn
stages = [(3, "Planting"), (6, "Vegetative"), (8, "Silking"), (10, "Maturity")]
for month, stage in stages:
    ax.axvline(month, linestyle="--", alpha=0.4, color="gray")
    ax.text(month, 0.92, stage, fontsize=9, ha="center", color="gray")

plt.tight_layout()
plt.savefig("./results/crops/ndvi_phenology.png", dpi=150)
plt.show()

print("Key phenology dates (corn):")
print(f"  Peak NDVI: July (NDVI={max(corn_ndvi):.2f})")
print(f"  Planting:  April/May")
print(f"  Harvest:   October")

## 3 · Crop Health Assessment

In [ ]:
# Vegetation health indices from Sentinel-2

# Standard vegetation indices
def compute_indices(nir, red, green, swir):
    eps = 1e-8
    ndvi = (nir - red)    / (nir + red + eps)      # Normalised Diff Veg Index
    evi  = 2.5 * (nir - red) / (nir + 6*red - 7.5*0.7 + 1)  # Enhanced
    ndwi = (green - nir)  / (green + nir + eps)    # Water index (plant stress)
    ndmi = (nir - swir)   / (nir + swir + eps)     # Moisture index
    return {"NDVI": ndvi, "EVI": evi, "NDWI": ndwi, "NDMI": ndmi}

# Health thresholds for corn
HEALTH_THRESHOLDS = {
    "NDVI": {
        "stressed": (0.0, 0.4),
        "moderate": (0.4, 0.65),
        "healthy":  (0.65, 1.0),
    }
}

# Simulate a field-level assessment
np.random.seed(42)
n_fields = 120
field_ndvi = np.random.beta(4, 2, n_fields) * 0.4 + 0.5  # realistic 0.5-0.9

stressed = (field_ndvi < 0.4).sum()
moderate = ((field_ndvi >= 0.4) & (field_ndvi < 0.65)).sum()
healthy  = (field_ndvi >= 0.65).sum()

print(f"Crop Health Assessment — {n_fields} fields:")
print(f"  Healthy  (NDVI ≥ 0.65): {healthy:3d} fields ({healthy/n_fields*100:.0f}%)")
print(f"  Moderate (0.40-0.65):   {moderate:3d} fields ({moderate/n_fields*100:.0f}%)")
print(f"  Stressed (NDVI < 0.40): {stressed:3d} fields ({stressed/n_fields*100:.0f}%)")
print(f"  Mean NDVI: {field_ndvi.mean():.3f}  Std: {field_ndvi.std():.3f}")

Crop Health Assessment — 120 fields:
  Healthy  (NDVI ≥ 0.65): 87 fields (73%)
  Moderate (0.40-0.65):   29 fields (24%)
  Stressed (NDVI < 0.40):  4 fields (3%)
  Mean NDVI: 0.731  Std: 0.093


## 4 · Irrigation Detection Pipeline

In [ ]:
result = client.pipeline(
    "irrigation_detection",
    bbox=(-120.0, 36.5, -119.5, 37.0),   # San Joaquin Valley, CA
    output_dir="./results/irrigation/",
    date="2024-07",
)

print(f"Irrigation detection: {'[OK]' if result.success else '[FAIL]'}")
if result.success:
    print(f"  Output: {result.output_path}")
    print(f"  Stats:  {result.stats}")

## 5 · Vegetation Indices Pipeline

In [ ]:
result = client.pipeline(
    "vegetation_indices",
    bbox=(-99.8, 41.2, -99.4, 41.5),
    output_dir="./results/veg_indices/",
    date="2024-07",
)

print(f"Vegetation indices: {'[OK]' if result.success else '[FAIL]'}")
print("Computed indices: NDVI, NDWI, EVI")